In [7]:
"""
Script COMBINADO para construir la tabla GHZ completa de un backend,
promediando dos tandas (v1 y v2):

    - De los archivos results_*.json (en V1_DIR y V2_DIR) saca: fidelity
    - De los archivos .pkl de calibracion (uno por tanda) RECONSTRUYE
      el circuito GHZ transpilado y mide: circuit_depth, two_qubit_gates
    - Promedia ambas tandas para cada tamaño de qubit
    - Calcula threshold (0.5 - epsilon) y Pass/Fail
    - Imprime la tabla final y la guarda en un CSV

IMPORTANTE:
    El circuit_depth / two_qubit_gates son una RECONSTRUCCION (se vuelve a
    transpilar el circuito GHZ contra la calibracion guardada en el .pkl),
    no el dato exacto registrado durante la ejecucion original. Es una muy
    buena aproximacion si el .pkl es de la MISMA fecha/sesion que los
    resultados (que es justamente el caso si usas los .pkl con el mismo
    sello de fecha que tus carpetas v1/v2).

USO:
    1. Rellena las 6 rutas de la seccion CONFIGURA (V1_DIR, V2_DIR,
       V1_TARGET_PKL, V2_TARGET_PKL, BACKEND_FILTER, EPSILON_ESPERADO)
    2. Ejecuta:  python build_full_ghz_table.py
    3. Te imprime la tabla final y guarda 'full_ghz_table.csv'
"""

import json
import os
import re
import glob
import csv
import pickle

from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager


# -------------------------------------------------------------------
# 1) CONFIGURA AQUI TUS RUTAS
# -------------------------------------------------------------------
V1_DIR = "./v1"   # carpeta con los results_*.json de la tanda 1
V2_DIR = "./v2"   # carpeta con los results_*.json de la tanda 2

V1_TARGET_PKL = "./v1/qubit_properties_ibm_pittsburgh_156q_20260330_123519.pkl"
V2_TARGET_PKL = "./v2/qubit_properties_ibm_pittsburgh_156q_20260330_140054.pkl"

BACKEND_FILTER = "ibm_pittsburgh"   # nombre del backend a procesar

GHZ_MODE = "lineal"          # ver utils/circuits.py si usasteis otro modo
OPTIMIZATION_LEVEL = 1       # nivel de optimizacion al transpilar

OUTPUT_CSV = "full_ghz_table.csv"


# -------------------------------------------------------------------
# 2) LECTURA LIGERA DE LOS results_*.json (sin cargar arrays pesados)
# -------------------------------------------------------------------
FILENAME_PATTERN = re.compile(r"results_(.+?)_(\d+)q_(\d{8}_\d{6})\.json")


def read_light_fields(filepath):
    fields = {"backend": None, "numero_qubits_inicial": None,
              "epsilon": None, "delta": None, "fidelity": None}

    with open(filepath, "r") as f:
        head = f.read(4000)

    def grab_scalar(key, cast=str):
        m = re.search(rf'"{key}":\s*([^,\n\}}]+)', head)
        if not m:
            return None
        val = m.group(1).strip().strip('"')
        try:
            return cast(val)
        except Exception:
            return val

    fields["backend"] = grab_scalar("backend")
    fields["numero_qubits_inicial"] = grab_scalar("numero_qubits_inicial", int)
    fields["epsilon"] = grab_scalar("epsilon", float)
    fields["delta"] = grab_scalar("delta", float)

    filesize = os.path.getsize(filepath)
    with open(filepath, "rb") as f:
        f.seek(max(0, filesize - 300))
        tail = f.read().decode(errors="ignore")
    m = re.search(r'"fidelity":\s*([^\s}]+)', tail)
    if m:
        fields["fidelity"] = float(m.group(1))
    else:
        with open(filepath, "r") as f:
            content = f.read()
        m = re.search(r'"fidelity":\s*([^\s,}]+)', content)
        if m:
            fields["fidelity"] = float(m.group(1))

    return fields


def scan_folder(folder):
    records = []
    for filepath in sorted(glob.glob(os.path.join(folder, "results_*.json"))):
        fname = os.path.basename(filepath)
        m = FILENAME_PATTERN.match(fname)
        if not m:
            continue
        backend_in_name, qubits_in_name, timestamp = m.groups()
        if BACKEND_FILTER and backend_in_name != BACKEND_FILTER:
            continue
        fields = read_light_fields(filepath)
        fields["file"] = fname
        fields["qubits_in_name"] = int(qubits_in_name)
        fields["timestamp"] = timestamp
        records.append(fields)
    return records


def fidelity_by_qubits(records):
    """{n_qubits: {"fidelity":..., "epsilon":...}} promediando duplicados internos."""
    by_n = {}
    for r in records:
        by_n.setdefault(r["qubits_in_name"], []).append(r)
    out = {}
    for n, recs in by_n.items():
        fids = [r["fidelity"] for r in recs if r["fidelity"] is not None]
        epss = [r["epsilon"] for r in recs if r["epsilon"] is not None]
        out[n] = {
            "fidelity": sum(fids) / len(fids) if fids else None,
            "epsilon": sum(epss) / len(epss) if epss else None,
        }
    return out


# -------------------------------------------------------------------
# 3) RECONSTRUCCION DEL CIRCUITO GHZ (igual que utils/circuits.py)
# -------------------------------------------------------------------
def create_lineal_ghz_circuit(num_qubits):
    qc = QuantumCircuit(num_qubits)
    qc.h(0)
    for i in range(num_qubits - 1):
        qc.cx(i, i + 1)
    return qc


def create_ghz_circuit(num_qubits, mode="lineal"):
    if mode == "lineal":
        return create_lineal_ghz_circuit(num_qubits)
    raise NotImplementedError(
        f"Modo '{mode}' no incluido en esta version reducida del script; "
        "copia las demas funciones de reconstruct_depth_2qgates.py si lo necesitas."
    )


def depth_and_gates(num_qubits, target):
    qc = create_ghz_circuit(num_qubits, mode=GHZ_MODE)
    pm = generate_preset_pass_manager(optimization_level=OPTIMIZATION_LEVEL, target=target)
    transpiled = pm.run(qc)
    depth = transpiled.depth()
    two_q = sum(1 for instr in transpiled.data if instr.operation.num_qubits == 2)
    return depth, two_q


# -------------------------------------------------------------------
# 4) PROGRAMA PRINCIPAL
# -------------------------------------------------------------------
def _avg(values):
    clean = [v for v in values if v is not None]
    return sum(clean) / len(clean) if clean else None


def main():
    # --- comprobaciones basicas ---
    for path, label in [(V1_DIR, "V1_DIR"), (V2_DIR, "V2_DIR")]:
        if not os.path.isdir(path):
            print(f"ERROR: {label} no existe: {path}")
            return
    for path, label in [(V1_TARGET_PKL, "V1_TARGET_PKL"), (V2_TARGET_PKL, "V2_TARGET_PKL")]:
        if not os.path.isfile(path):
            print(f"ERROR: {label} no existe: {path}")
            return

    # --- fidelidad desde los JSON ---
    print(f"Leyendo fidelidad de v1: {V1_DIR}")
    v1_fid = fidelity_by_qubits(scan_folder(V1_DIR))
    print(f"  -> tamaños encontrados: {sorted(v1_fid.keys())}")

    print(f"Leyendo fidelidad de v2: {V2_DIR}")
    v2_fid = fidelity_by_qubits(scan_folder(V2_DIR))
    print(f"  -> tamaños encontrados: {sorted(v2_fid.keys())}")

    all_qubit_sizes = sorted(set(v1_fid.keys()) | set(v2_fid.keys()))
    if not all_qubit_sizes:
        print("ERROR: no se encontro ningun tamaño de qubit en v1 ni v2.")
        return

    # --- cargar los dos targets de calibracion ---
    print(f"\nCargando target v1: {V1_TARGET_PKL}")
    with open(V1_TARGET_PKL, "rb") as f:
        target_v1 = pickle.load(f)
    print(f"Cargando target v2: {V2_TARGET_PKL}")
    with open(V2_TARGET_PKL, "rb") as f:
        target_v2 = pickle.load(f)

    # --- construir filas ---
    rows = []
    print(f"\n{'Qubits':>7} | {'Fid v1':>8} | {'Fid v2':>8} | {'MeanFid':>8} | "
          f"{'Depth v1':>9} | {'Depth v2':>9} | {'MeanDep':>8} | "
          f"{'2Qg v1':>7} | {'2Qg v2':>7} | {'Mean2Qg':>8} | {'Thresh':>7} | Pass/Fail")
    print("-" * 130)

    for n in all_qubit_sizes:
        fid_v1 = v1_fid.get(n, {}).get("fidelity")
        fid_v2 = v2_fid.get(n, {}).get("fidelity")
        mean_fid = _avg([fid_v1, fid_v2])

        eps_v1 = v1_fid.get(n, {}).get("epsilon")
        eps_v2 = v2_fid.get(n, {}).get("epsilon")
        epsilon = _avg([eps_v1, eps_v2])
        threshold = (0.5 - epsilon) if epsilon is not None else None

        # Reconstruir depth/2Q gates SOLO si ese n existe en la tanda correspondiente
        depth_v1 = gates_v1 = None
        if n in v1_fid:
            depth_v1, gates_v1 = depth_and_gates(n, target_v1)

        depth_v2 = gates_v2 = None
        if n in v2_fid:
            depth_v2, gates_v2 = depth_and_gates(n, target_v2)

        mean_depth = _avg([depth_v1, depth_v2])
        mean_gates = _avg([gates_v1, gates_v2])

        passed = (mean_fid is not None and threshold is not None and mean_fid > threshold)

        def fmt(x, nd=4):
            return f"{x:.{nd}f}" if isinstance(x, float) else ("--" if x is None else str(x))

        print(f"{n:>7} | {fmt(fid_v1):>8} | {fmt(fid_v2):>8} | {fmt(mean_fid):>8} | "
              f"{fmt(depth_v1,0):>9} | {fmt(depth_v2,0):>9} | {fmt(mean_depth,1):>8} | "
              f"{fmt(gates_v1,0):>7} | {fmt(gates_v2,0):>7} | {fmt(mean_gates,1):>8} | "
              f"{fmt(threshold,2):>7} | {'Pass' if passed else 'Fail'}")

        rows.append({
            "qubits": n,
            "fidelity_v1": fid_v1, "fidelity_v2": fid_v2, "mean_fidelity": mean_fid,
            "depth_v1": depth_v1, "depth_v2": depth_v2, "mean_depth": mean_depth,
            "two_qubit_gates_v1": gates_v1, "two_qubit_gates_v2": gates_v2,
            "mean_two_qubit_gates": mean_gates,
            "epsilon": epsilon, "threshold": threshold,
            "pass_fail": "Pass" if passed else "Fail",
        })

    with open(OUTPUT_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nTabla guardada en: {OUTPUT_CSV}")
    print("\nRECUERDA: Depth y 2Q gates son una RECONSTRUCCION del circuito")
    print("transpilado contra la calibracion guardada en el .pkl, no el dato")
    print("exacto registrado durante la ejecucion original.")


if __name__ == "__main__":
    main()

Leyendo fidelidad de v1: ./v1
  -> tamaños encontrados: [16, 23, 27, 28, 29]
Leyendo fidelidad de v2: ./v2
  -> tamaños encontrados: [16, 23, 27, 28, 29]

Cargando target v1: ./v1/qubit_properties_ibm_pittsburgh_156q_20260330_123519.pkl
Cargando target v2: ./v2/qubit_properties_ibm_pittsburgh_156q_20260330_140054.pkl

 Qubits |   Fid v1 |   Fid v2 |  MeanFid |  Depth v1 |  Depth v2 |  MeanDep |  2Qg v1 |  2Qg v2 |  Mean2Qg |  Thresh | Pass/Fail
----------------------------------------------------------------------------------------------------------------------------------
     16 |   0.6748 |   0.6112 |   0.6430 |        63 |        63 |     63.0 |      15 |      15 |     15.0 |    0.40 | Pass
     23 |   0.5974 |   0.5853 |   0.5913 |        91 |        91 |     91.0 |      22 |      22 |     22.0 |    0.40 | Pass
     27 |   0.4229 |   0.4680 |   0.4454 |       107 |       107 |    107.0 |      26 |      26 |     26.0 |    0.40 | Pass
     28 |   0.3628 |   0.4904 |   0.4266 |      

In [9]:


import os
import re
import glob
import pickle
import statistics
from datetime import datetime


# -------------------------------------------------------------------
# 1) CONFIGURA AQUI
# -------------------------------------------------------------------
V1_DIR = "./v1"
V2_DIR = "./v2"
BACKEND_FILTER = "ibm_pittsburgh"

# Si solo tienes un .pkl (no dos, uno por tanda), deja V2 igual a V1 o
# vacio; el script se adapta.

PKL_PATTERN = re.compile(r"qubit_properties_(.+?)_(\d+)q_(\d{8}_\d{6})\.pkl")


# -------------------------------------------------------------------
# 2) LOCALIZAR LOS .pkl DEL BACKEND DENTRO DE UNA CARPETA
# -------------------------------------------------------------------
def find_calibration_pkls(folder, backend_filter):
    found = []
    if not os.path.isdir(folder):
        return found
    for filepath in sorted(glob.glob(os.path.join(folder, "qubit_properties_*.pkl"))):
        fname = os.path.basename(filepath)
        m = PKL_PATTERN.match(fname)
        if not m:
            continue
        backend_in_name, qubits, timestamp = m.groups()
        if backend_in_name != backend_filter:
            continue
        found.append({
            "file": fname, "filepath": filepath,
            "qubits": int(qubits), "timestamp": timestamp,
        })
    found.sort(key=lambda d: d["timestamp"])
    return found


def format_timestamp(ts):
    """20260330_123519 -> 2026-03-30 12:35:19"""
    try:
        dt = datetime.strptime(ts, "%Y%m%d_%H%M%S")
        return dt.strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return ts


# -------------------------------------------------------------------
# 3) EXTRAER METADATOS DE UN .pkl (objeto Target de Qiskit)
# -------------------------------------------------------------------
def extract_metadata(pkl_path):
    with open(pkl_path, "rb") as f:
        target = pickle.load(f)

    meta = {}
    meta["num_qubits"] = target.num_qubits
    meta["operation_names"] = sorted(target.operation_names)

    # --- T1 / T2 medianos ---
    qp = target.qubit_properties
    t1_values = [q.t1 for q in qp if q.t1 is not None] if qp else []
    t2_values = [q.t2 for q in qp if q.t2 is not None] if qp else []
    meta["median_t1_us"] = (statistics.median(t1_values) * 1e6) if t1_values else None
    meta["median_t2_us"] = (statistics.median(t2_values) * 1e6) if t2_values else None

    # --- topologia: grado medio de conectividad ---
    cmap = target.build_coupling_map()
    if cmap is not None:
        n_edges = cmap.size()
        n_qubits = target.num_qubits
        avg_degree = (2 * n_edges) / n_qubits if n_qubits else None
        meta["avg_connectivity_degree"] = avg_degree
        if avg_degree is not None:
            if avg_degree < 2.3:
                meta["topology_guess"] = "Heavy-hex (low connectivity, ~2 per qubit)"
            elif avg_degree < 3.3:
                meta["topology_guess"] = "Square / heavy-square (~3 per qubit)"
            elif avg_degree < 4.5:
                meta["topology_guess"] = "Square lattice / grid (~4 per qubit)"
            else:
                meta["topology_guess"] = "High connectivity / all-to-all-like"
    else:
        meta["avg_connectivity_degree"] = None
        meta["topology_guess"] = "Unknown (no coupling map found)"

    # --- gate nativa de 2 qubits: buscamos la primera de 2 qargs que conozcamos ---
    two_q_gate_name = None
    for name in target.operation_names:
        try:
            qargs_list = [q for q in target[name].keys() if q is not None]
        except Exception:
            continue
        if qargs_list and len(qargs_list[0]) == 2:
            two_q_gate_name = name
            break

    if two_q_gate_name:
        props = target[two_q_gate_name]
        errors = [p.error for q, p in props.items() if q is not None and p is not None and p.error is not None]
        durations = [p.duration for q, p in props.items() if q is not None and p is not None and p.duration is not None]
        meta["two_qubit_gate_name"] = two_q_gate_name
        meta["two_qubit_gate_median_error"] = statistics.median(errors) if errors else None
        meta["two_qubit_gate_median_duration_ns"] = (
            statistics.median(durations) * 1e9 if durations else None
        )
    else:
        meta["two_qubit_gate_name"] = None
        meta["two_qubit_gate_median_error"] = None
        meta["two_qubit_gate_median_duration_ns"] = None

    # --- gate nativa de 1 qubit mas comun (ej. sx, x, rz) ---
    one_q_gate_name = None
    for candidate in ["sx", "x", "rz", "id"]:
        if candidate in target.operation_names:
            try:
                qargs_list = [q for q in target[candidate].keys() if q is not None]
            except Exception:
                continue
            if qargs_list and len(qargs_list[0]) == 1:
                one_q_gate_name = candidate
                break

    if one_q_gate_name:
        props = target[one_q_gate_name]
        errors = [p.error for q, p in props.items() if q is not None and p is not None and p.error is not None]
        meta["one_qubit_gate_name"] = one_q_gate_name
        meta["one_qubit_gate_median_error"] = statistics.median(errors) if errors else None
    else:
        meta["one_qubit_gate_name"] = None
        meta["one_qubit_gate_median_error"] = None

    return meta


# -------------------------------------------------------------------
# 4) PROGRAMA PRINCIPAL
# -------------------------------------------------------------------
def _avg(values):
    clean = [v for v in values if v is not None]
    return sum(clean) / len(clean) if clean else None


def report_for_folder(label, folder):
    print(f"\n--- {label}: {folder} ---")
    pkls = find_calibration_pkls(folder, BACKEND_FILTER)
    if not pkls:
        print(f"  No se encontro ningun qubit_properties_{BACKEND_FILTER}_*.pkl aqui.")
        return None

    # Si hay varios, usamos el mas reciente de esa carpeta
    chosen = pkls[-1]
    print(f"  Usando: {chosen['file']}  (fecha: {format_timestamp(chosen['timestamp'])})")
    if len(pkls) > 1:
        print(f"  (Aviso: habia {len(pkls)} archivos de este backend en esta carpeta; "
              f"se usa el mas reciente. Otros encontrados: "
              f"{[p['file'] for p in pkls if p != chosen]})")

    meta = extract_metadata(chosen["filepath"])
    meta["timestamp"] = chosen["timestamp"]
    return meta


def main():
    meta_v1 = report_for_folder("v1", V1_DIR)
    meta_v2 = report_for_folder("v2", V2_DIR)

    if meta_v1 is None and meta_v2 is None:
        print(f"\nERROR: no se encontro ningun .pkl de '{BACKEND_FILTER}' en v1 ni v2.")
        return

    print("\n" + "=" * 70)
    print("RESUMEN PARA LA TABLA SPEQC (bloque Hardware)")
    print("=" * 70)

    # --- campos que no cambian entre v1/v2 (cogemos el que exista) ---
    ref = meta_v1 or meta_v2
    print(f"Total qubits:            {ref['num_qubits']}")
    print(f"Native gates:            {', '.join(ref['operation_names'])}")
    print(f"Topology (heuristic):    {ref['topology_guess']}  "
          f"(avg degree = {ref['avg_connectivity_degree']:.2f})")

    # --- gate de 2 qubits ---
    g2 = ref["two_qubit_gate_name"]
    if g2:
        err = ref["two_qubit_gate_median_error"]
        dur = ref["two_qubit_gate_median_duration_ns"]
        print(f"2Q native gate:          {g2}  "
              f"(median error = {err*100:.3f}% , median duration = {dur:.1f} ns)")

    g1 = ref["one_qubit_gate_name"]
    if g1:
        err1 = ref["one_qubit_gate_median_error"]
        print(f"1Q native gate:          {g1}  (median error = {err1*100:.4f}%)")

    # --- T1 / T2, promediando v1 y v2 si ambos existen ---
    t1_v1 = meta_v1["median_t1_us"] if meta_v1 else None
    t1_v2 = meta_v2["median_t1_us"] if meta_v2 else None
    t2_v1 = meta_v1["median_t2_us"] if meta_v1 else None
    t2_v2 = meta_v2["median_t2_us"] if meta_v2 else None

    mean_t1 = _avg([t1_v1, t1_v2])
    mean_t2 = _avg([t2_v1, t2_v2])

    print()
    def fmt_us(x):
        return f"{x:.1f} us" if x is not None else "--"

    print(f"Median T1:               v1={fmt_us(t1_v1)}  v2={fmt_us(t1_v2)}  "
          f"-> MEAN = {fmt_us(mean_t1)}")
    print(f"Median T2:               v1={fmt_us(t2_v1)}  v2={fmt_us(t2_v2)}  "
          f"-> MEAN = {fmt_us(mean_t2)}")

    if meta_v1:
        print(f"\nFecha calibracion v1:    {format_timestamp(meta_v1['timestamp'])}")
    if meta_v2:
        print(f"Fecha calibracion v2:    {format_timestamp(meta_v2['timestamp'])}")

    print("\n" + "-" * 70)
    print("CAMPOS QUE NO SE PUEDEN SACAR DEL .pkl (rellenar a mano si los teneis):")
    print("  - Calibration time required  (no existe en el objeto Target)")
    print("  - Shots per circuit          (parametro de ejecucion en main.py)")
    print("-" * 70)


if __name__ == "__main__":
    main()


--- v1: ./v1 ---
  Usando: qubit_properties_ibm_pittsburgh_156q_20260330_123519.pkl  (fecha: 2026-03-30 12:35:19)

--- v2: ./v2 ---
  Usando: qubit_properties_ibm_pittsburgh_156q_20260330_140054.pkl  (fecha: 2026-03-30 14:00:54)

RESUMEN PARA LA TABLA SPEQC (bloque Hardware)
Total qubits:            156
Native gates:            cz, delay, id, if_else, measure, measure_2, reset, rz, sx, x
Topology (heuristic):    Heavy-hex (low connectivity, ~2 per qubit)  (avg degree = 2.00)
2Q native gate:          cz  (median error = 0.170% , median duration = 88.0 ns)
1Q native gate:          sx  (median error = 0.0204%)

Median T1:               v1=298.0 us  v2=298.0 us  -> MEAN = 298.0 us
Median T2:               v1=328.8 us  v2=328.8 us  -> MEAN = 328.8 us

Fecha calibracion v1:    2026-03-30 12:35:19
Fecha calibracion v2:    2026-03-30 14:00:54

----------------------------------------------------------------------
CAMPOS QUE NO SE PUEDEN SACAR DEL .pkl (rellenar a mano si los teneis):
  - Cali